# 1. Identificação e Descrição do Problema

Título: Reconhecimento de 12 selos de mão do anime Naruto.

Integrantes: Luiz Cristóvão Rezende Poderoso, Jean Fonseca Santos e Ivanilton Vieira dos Santos.

Fonte dos Dados: [Naruto Hand Sign Dataset](https://www.kaggle.com/datasets/vikranthkanumuru/naruto-hand-sign-dataset). 

Objetivo: Reconhecer os 12 selos de mão do anime Naruto em imagens.

Atributo-Alvo: Categoria do selo de mão.

Atributos Preditivos: Matriz de pixels da imagem.

Tipo da Tarefa: Classificação.


# 2. Compreensão dos Dados



Instalação de bibliotecas necessárias para análise e comparação das imagens.

In [ ]:
!pip install Pillow ImageHash

Agora, importaremos as bibliotecas e declararemos as constantes necessárias.

In [ ]:
import os
from PIL import Image
import imagehash

DATASET_PATH = os.path.join("../../dataset")
SIGNALS = ["bird", "boar", "dog", "dragon", "hare", "horse", "monkey", "ox", "ram", "rat", "snake", "tiger"]
IMG_DIFFERENCE_LIMIT = 5

Abaixo, declaramos as métricas gerais para análise do dataset.

In [ ]:
resolutions_count = dict()
total_count = 0
png_count = 0
jpg_count = 0
duplicates_count = 0
simmilar_count = 0

O objetivo do trecho seguinte é obter as informações acerca dos formatos das imagens, resolução e, utilizando a biblioteca `imagehash`, verificar se a imagem é duplicada ou extremamente semelhante a outra analisada anteriormente (Distância de Hamming <= 5). A análise, por enquanto, é agrupada por selo de mão.

In [ ]:
for signal in os.listdir(DATASET_PATH):
    signal_path = os.path.join(DATASET_PATH, signal)

    if (signal not in SIGNALS): 
        continue
    
    signal_total_count = 0
    signal_png_count = 0
    signal_jpg_count = 0
    signal_duplicates_count = 0
    signal_simmilar_count = 0

    signal_resolutions_count = dict()
    seen_hashes = set()

    for img_filename in os.listdir(signal_path):
        lower_img_name = img_filename.lower()

        # formato
        if lower_img_name.endswith(".png"): signal_png_count += 1
        elif lower_img_name.endswith(".jpg") or lower_img_name.endswith(".jpeg"): signal_jpg_count += 1
        signal_total_count += 1

        img_path = os.path.join(signal_path, img_filename)
        try:
            with Image.open(img_path) as img:
                # resoluções
                key = f"{img.size[0]}x{img.size[1]}"
                
                if key not in signal_resolutions_count: signal_resolutions_count[key] = 0
                if key not in resolutions_count: resolutions_count[key] = 0
                
                signal_resolutions_count[key] += 1
                resolutions_count[key] += 1
                
                # duplicatas / semelhantes
                img_hash = imagehash.phash(img)
                
                is_duplicate = False
                is_similar = False

                for seen_hash in seen_hashes:
                    difference = img_hash - seen_hash
                    
                    if difference == 0:
                        is_duplicate = True
                        break
                    elif difference <= IMG_DIFFERENCE_LIMIT:
                        is_similar = True
                
                if is_duplicate:
                    signal_duplicates_count += 1
                elif is_similar:
                    signal_simmilar_count += 1
                else:
                    seen_hashes.add(img_hash)
        except Exception:
            pass

    print(f"{'-' * 25}\nSelo: {signal}\nTotal de Imagens: {signal_total_count}")

    print("\n---RESOLUÇÕES---")
    for key in signal_resolutions_count:
        length = signal_resolutions_count[key]
        print(f"{key}: {length}")

    print(f"\n---FORMATO---\nImagens PNG: {signal_png_count}\nImagens JPG: {signal_jpg_count}\nOutras imagens: {signal_total_count - signal_png_count - signal_jpg_count}")

    print(f"\n---GERAL---\nDuplicatas: {signal_duplicates_count}\nSemelhantes: {signal_simmilar_count}\n{'-' * 25}\n")
    
    total_count += signal_total_count
    png_count += signal_png_count
    jpg_count += signal_jpg_count
    duplicates_count += signal_duplicates_count
    simmilar_count += signal_simmilar_count

Por último, para obter a análise geral aceca das imagens:

In [ ]:
print(f"{'-' * 25}\nTotal\nTotal de Imagens: {total_count}")

print("\n---RESOLUÇÕES---")
for key in resolutions_count:
    length = resolutions_count[key]
    print(f"{key}: {length}")
    
print(f"\n---FORMATO---\nImagens PNG: {png_count}\nImagens JPG: {jpg_count}\nOutras imagens: {total_count - png_count - jpg_count}")

print(f"\n---GERAL---\nDuplicatas: {duplicates_count}\nSemelhantes: {simmilar_count}\n{'-' * 25}\n")

O relatório final aponta a existência de 1960 imagens no dataset original. As imagens estão em 3 resoluções diferentes: `640x480`, `1280x720` e `3264x1840`, sendo majoritariamente (1620) em `1280x720`. Ademais, as imagens, em sua maioria (1619 imagens), são `PNG`, já o restante em `JPG`. Além disso, o dataset possui uma quantidade preocupante de imagens duplicadas (Distância de Hamming igual a 0): 420, e uma ainda mais preocupante de semelhantes (Distância de Hamming menor ou igual a 5): 1189.

# 3. Analise Exploratória

# 4. Pré-processamento

## 4.1 Novas imagens

As imagens do dataset, além de escassas, apresentaram um baixo padrão de qualidade, sendo muitos delas frames consecutivos de um mesmo vídeo (quase nenhuma variação entre elas). Devido a isso, foi necessário adicionar novas imagens ao conjunto. Primeiro, a equipe desenvolveu um script que extrai imagens de um vídeo a cada 30 frames (veja a pasta `scripts/extracao`), já inserindo-as nas pastas corretas do dataset. Os vídeos foram gravados de forma com que diversas rotações e posições de um mesmo selo fossem obtidos. Depois, os integrantes da equipe gravaram vídeos fazendo os selos de mão em diferentes ambientes com diferentes iluminações. Também, foi indispensável a ajuda de outras pessoas na gravação desses vídeos, gerando diversidade suficiente para um modelo proveitoso. É fundamental destacar que houve revisão de cada uma das imagens, visando excluir a presença de registros borrados ou de baixa qualidade.

## 4.2 Redimensionamento das imagens

Antes de irem para o modelo, as imagens utilizadas são redimensionadas para uma matriz fixa de pixels, com o objetivo de uniformizar os dados.

## 4.3 Planificação das matrizes

As matrizes foram reorganizadas de modo que se tornassem matrizes unidimensionais, facilitando a utilização delas nos algoritmos requisitados.

# 5. Separação dos Dados

Os dados foram separados com a seguinte proporção: 80% para os dados de treinamento e 20% para os dados de teste. Essa proporção foi escolhida por ser a padrão para treinamento de classificação de imagens para a quantidade de dados que possuímos. 

Para seguir essa divisão, os dados são divididos em subpastas dentro dos sinais de mão, onde cada subpasta é nomeada como uma identificador único por vídeo de origem. Assim, 80% das subpastas são utilizadas para treinamento e 20% para teste. Isso evita que imagens de um mesmo vídeo sejam utilizadas para teste e treinamento simultânemante, o que poderia gerar uma taxa de acerto alta artificialmente devido à semelhança dos dados. 

# 6. Modelagem

# 7. Avaliação e Discussão